## Pizza Service (Taskmaster-1, übersetzt für EMNLP 2020)

Dataset from M Amin Farajian, António V Lopes, André FT Martins, Sameen Maruf, Gholamreza Haffari (2020). Findings of the wmt 2020 shared task on chat translation (https://aclanthology.org/2020.wmt-1.3/)

First we download the dataset from github:

In [ ]:
!git clone https://github.com/Unbabel/BConTrasT BConTrasT

Let’s load the JSON files and take an initial, unrefined look at the dataset. Since we do not intend to train a machine learning model, we will use the train, development, and test sets collectively:

In [ ]:
import json
import os
import pandas as pd

DATASET_FOLDER = f'..{os.path.sep}data{os.path.sep}BConTrasT'
DATASET_FILES = ['train.json', 'dev.json', 'test1_corpus-with_gold_references.json']

dict_dataset = {}
for ds_file in DATASET_FILES:
    with open(os.path.join(DATASET_FOLDER, ds_file), encoding="utf-8") as file:
        dict_dataset.update(json.load(file))

dict_dataset

A useful first step when exploring a corpus is to determine how many dialogs it contains:

In [ ]:
print(f'Number of dialogs: {len(dict_dataset)}')

A pandas DataFrame offers a powerful, flexible, and efficient way to store, manipulate, and analyze structured tabular data in Python:

In [ ]:
utterance_list = []
for dialogID, dialog_steps in dict_dataset.items():
    for ds in dialog_steps:
        dialog_step_dict = {
            'dialogID': dialogID, 
            'utteranceID': ds['utteranceID'],
            'speaker': ds['speaker'],
            'dt': ds['source'] if ds['speaker'] == 'customer' else ds['target'],
            'en': ds['target'] if ds['speaker'] == 'customer' else ds['source']
        }
        utterance_list.append(dialog_step_dict)

# to flatten the data we create a list of dicts for each row and add this list en-bloc to the pandas dataframe. Using the append or concat function of pandas might work but is very inefficient as the data is copied for each call. 
df = pd.DataFrame(utterance_list)
df.head(5)

The BConTrasT dataset includes six different domains, one of which is pizza ordering. Since the dataset does not specify which dialogs belong to each domain, we filter utterances that contain the keyword pizza:

In [ ]:
# adding a column if the dialog step contains the word pizza or Pizza
df['pizza_domain'] = df['dt'].apply(lambda x: True if 'pizza' in x.lower() else False)
dialogs_pizza_domain = list(df[df['pizza_domain']]['dialogID'].drop_duplicates())
print("Number of dialogs in pizza domain: ", len(dialogs_pizza_domain))

If we are interested in how users phrase their orders for pizza, we can select the utterances that contain the keyword ___pizza___

In [ ]:
df[df['pizza_domain']]

For a complete example of a pizza-ordering dialog, see:

In [ ]:
df[df['dialogID'] == 'dlg-ed5440cf-4f96-4f53-8290-0a462803c2b8']